In [1]:
"""
PR1 unit tests — KV cache plumbing equivalence.

For each level (attention, encoder, module), assert that
    one_shot(x[:, :T]) == cat(prefill(x[:, :P]), decode(x[:, P:], past_kv=cache_from_prefill))
"""
import torch
import torch.nn.functional as F
from functools import partial

from uni2ts.module.attention import GroupedQueryAttention
from uni2ts.module.transformer import TransformerEncoder
from uni2ts.module.norm import RMSNorm
from uni2ts.module.position import (
    BinaryAttentionBias,
    QueryKeyProjection,
    RotaryProjection,
)
from uni2ts.model.moiraic.module import MoiraicModule

torch.manual_seed(0)
device = "cuda:7" if torch.cuda.is_available() else "cpu"
ATOL = 1e-4


def causal_mask(T, device):
    return torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))


In [2]:
# ---------------------------------------------------------------------------
# Step 1 — GroupedQueryAttention, no biases / no qk_proj
# ---------------------------------------------------------------------------
def test_attention_basic():
    d_model, num_heads, num_groups = 64, 4, 2
    attn = (
        GroupedQueryAttention(
            dim=d_model, num_heads=num_heads, num_groups=num_groups,
            bias=False, norm_layer=None,
        )
        .to(device).eval()
    )

    B, T, P = 2, 12, 7
    x = torch.randn(B, T, d_model, device=device)
    full_mask = causal_mask(T, device).unsqueeze(0).expand(B, -1, -1)

    with torch.no_grad():
        out_full = attn(x, x, x, attn_mask=full_mask)

        out_pref, cache = attn(
            x[:, :P], x[:, :P], x[:, :P],
            attn_mask=full_mask[:, :P, :P],
            return_kv=True,
        )

        out_suf, _ = attn(
            x[:, P:], x[:, P:], x[:, P:],
            attn_mask=full_mask[:, P:, :],   # [B, suffix_len, T]
            past_kv=cache,
            return_kv=True,
        )

    assert torch.allclose(out_pref, out_full[:, :P], atol=ATOL), \
        f"prefix max diff = {(out_pref - out_full[:, :P]).abs().max().item():.2e}"
    assert torch.allclose(out_suf, out_full[:, P:], atol=ATOL), \
        f"suffix max diff = {(out_suf - out_full[:, P:]).abs().max().item():.2e}"
    print("✓ test_attention_basic")

test_attention_basic()

✓ test_attention_basic


In [3]:
# ---------------------------------------------------------------------------
# Step 2 — GroupedQueryAttention with RoPE + var bias + qk-norm (Moiraic-like)
# ---------------------------------------------------------------------------
def test_attention_full():
    d_model, num_heads, num_groups, max_seq_len = 64, 4, 2, 64
    attn = (
        GroupedQueryAttention(
            dim=d_model, num_heads=num_heads, num_groups=num_groups,
            bias=False, norm_layer=RMSNorm,
            var_attn_bias=partial(
                BinaryAttentionBias,
                dim=d_model, num_heads=num_heads, num_groups=num_groups,
            ),
            time_qk_proj=partial(
                QueryKeyProjection,
                dim=d_model, num_heads=num_heads, num_groups=num_groups,
                proj_layer=RotaryProjection,
                kwargs=dict(max_len=max_seq_len),
                partial_factor=(0.0, 0.5),
            ),
        )
        .to(device).eval()
    )

    B, T, P = 2, 12, 7
    x = torch.randn(B, T, d_model, device=device)
    var_id = torch.zeros(B, T, dtype=torch.long, device=device)
    time_id = torch.arange(T, device=device).unsqueeze(0).expand(B, -1).contiguous()
    full_mask = causal_mask(T, device).unsqueeze(0).expand(B, -1, -1)

    with torch.no_grad():
        out_full = attn(
            x, x, x, attn_mask=full_mask,
            query_var_id=var_id, kv_var_id=var_id,
            query_time_id=time_id, kv_time_id=time_id,
        )

        out_pref, cache = attn(
            x[:, :P], x[:, :P], x[:, :P],
            attn_mask=full_mask[:, :P, :P],
            query_var_id=var_id[:, :P], kv_var_id=var_id[:, :P],
            query_time_id=time_id[:, :P], kv_time_id=time_id[:, :P],
            return_kv=True,
        )

        out_suf, _ = attn(
            x[:, P:], x[:, P:], x[:, P:],
            attn_mask=full_mask[:, P:, :],
            query_var_id=var_id[:, P:], kv_var_id=var_id[:, P:],
            query_time_id=time_id[:, P:], kv_time_id=time_id[:, P:],
            past_kv=cache,
            past_kv_var_id=var_id[:, :P],
            past_kv_time_id=time_id[:, :P],
            return_kv=True,
        )

    assert torch.allclose(out_pref, out_full[:, :P], atol=ATOL), \
        f"prefix max diff = {(out_pref - out_full[:, :P]).abs().max().item():.2e}"
    assert torch.allclose(out_suf, out_full[:, P:], atol=ATOL), \
        f"suffix max diff = {(out_suf - out_full[:, P:]).abs().max().item():.2e}"
    print("✓ test_attention_full")


test_attention_full()

✓ test_attention_full


In [4]:
# ---------------------------------------------------------------------------
# Step 3 — TransformerEncoder with the exact Moiraic config
# ---------------------------------------------------------------------------
def test_encoder_cache():
    d_model, d_ff, num_layers, max_seq_len = 64, 128, 2, 64
    enc = (
        TransformerEncoder(
            d_model=d_model, num_layers=num_layers,
            num_heads=None, pre_norm=True,
            attn_dropout_p=0.0, dropout_p=0.0,
            norm_layer=RMSNorm, activation=F.silu,
            use_glu=True, use_qk_norm=True,
            var_attn_bias_layer=partial(BinaryAttentionBias),
            time_qk_proj_layer=partial(
                QueryKeyProjection,
                proj_layer=RotaryProjection,
                kwargs=dict(max_len=max_seq_len),
                partial_factor=(0.0, 0.5),
            ),
            shared_var_attn_bias=False,
            shared_time_qk_proj=True,
            d_ff=d_ff,
        )
        .to(device).eval()
    )

    B, T, P = 2, 12, 7
    x = torch.randn(B, T, d_model, device=device)
    var_id = torch.zeros(B, T, dtype=torch.long, device=device)
    time_id = torch.arange(T, device=device).unsqueeze(0).expand(B, -1).contiguous()
    full_mask = causal_mask(T, device).unsqueeze(0).expand(B, -1, -1)

    with torch.no_grad():
        out_full = enc(x, full_mask, var_id=var_id, time_id=time_id)

        out_pref, layer_kvs = enc(
            x[:, :P], full_mask[:, :P, :P],
            var_id=var_id[:, :P], time_id=time_id[:, :P],
            return_kvs=True,
        )

        out_suf, _ = enc(
            x[:, P:], full_mask[:, P:, :],
            var_id=var_id[:, P:], time_id=time_id[:, P:],
            past_kvs=layer_kvs,
            past_kv_var_id=var_id[:, :P],
            past_kv_time_id=time_id[:, :P],
            return_kvs=True,
        )

    diff_pref = (out_pref - out_full[:, :P]).abs().max().item()
    diff_suf = (out_suf - out_full[:, P:]).abs().max().item()
    assert diff_pref < ATOL, f"encoder prefix max diff = {diff_pref:.2e}"
    assert diff_suf < ATOL, f"encoder suffix max diff = {diff_suf:.2e}"

    assert len(layer_kvs) == num_layers
    for k, v in layer_kvs:
        assert k.shape[-2] == P and v.shape[-2] == P, \
            f"cached len {k.shape[-2]} != {P}"
    print("✓ test_encoder_cache")


test_encoder_cache()

✓ test_encoder_cache


In [5]:
# ---------------------------------------------------------------------------
# Step 4 — MoiraicModule(return_cache=True) is a no-op for predictions
#          and produces a structurally correct cache
# ---------------------------------------------------------------------------
def test_moiraic_module_return_cache():
    module = (
        MoiraicModule(
            d_model=64, d_ff=128, num_layers=2,
            patch_size=8, max_seq_len=64,
            attn_dropout_p=0.0, dropout_p=0.0,
            scaling=True, num_predict_token=1,
        )
        .to(device).eval()
    )

    B, S, P = 2, 10, 8
    target = torch.randn(B, S, P, device=device)
    observed = torch.ones(B, S, P, dtype=torch.bool, device=device)
    sample_id = torch.zeros(B, S, dtype=torch.long, device=device)
    time_id = torch.arange(S, device=device).unsqueeze(0).expand(B, -1).contiguous()
    variate_id = torch.zeros(B, S, dtype=torch.long, device=device)
    pred_mask = torch.zeros(B, S, dtype=torch.bool, device=device)
    pred_mask[:, -2:] = True  # last two tokens are the prediction window

    with torch.no_grad():
        preds_no_cache = module(
            target, observed, sample_id, time_id, variate_id, pred_mask,
            training_mode=False,
        )
        preds_cached, cache = module(
            target, observed, sample_id, time_id, variate_id, pred_mask,
            training_mode=False, return_cache=True,
        )

    assert torch.allclose(preds_no_cache, preds_cached, atol=ATOL), \
        "return_cache=True must not change predictions"

    assert set(cache.keys()) >= {
        "layer_kvs", "kv_var_id", "kv_time_id", "kv_sample_id",
        "prediction_mask", "loc", "scale",
    }
    assert len(cache["layer_kvs"]) == 2
    for k, v in cache["layer_kvs"]:
        assert k.shape[-2] == S and v.shape[-2] == S
    assert cache["loc"].shape == (B, S, 1)
    assert cache["scale"].shape == (B, S, 1)
    print("✓ test_moiraic_module_return_cache")


test_moiraic_module_return_cache()

✓ test_moiraic_module_return_cache


In [6]:
# ---------------------------------------------------------------------------
# Step 5 — Backward compatibility: existing call sites are unchanged
# ---------------------------------------------------------------------------
def test_backward_compat_no_kwargs():
    # Smoke test: calling with the old signature (no PR1 kwargs) still works
    # and produces the same result as before.
    d_model = 64
    enc = (
        TransformerEncoder(
            d_model=d_model, num_layers=2, num_heads=None, pre_norm=True,
            attn_dropout_p=0.0, dropout_p=0.0,
            norm_layer=RMSNorm, activation=F.silu,
            use_glu=True, use_qk_norm=True,
            var_attn_bias_layer=partial(BinaryAttentionBias),
            time_qk_proj_layer=partial(
                QueryKeyProjection,
                proj_layer=RotaryProjection,
                kwargs=dict(max_len=64),
                partial_factor=(0.0, 0.5),
            ),
            shared_var_attn_bias=False, shared_time_qk_proj=True, d_ff=128,
        )
        .to(device).eval()
    )
    B, T = 2, 10
    x = torch.randn(B, T, d_model, device=device)
    var_id = torch.zeros(B, T, dtype=torch.long, device=device)
    time_id = torch.arange(T, device=device).unsqueeze(0).expand(B, -1).contiguous()
    mask = causal_mask(T, device).unsqueeze(0).expand(B, -1, -1)

    with torch.no_grad():
        # Old-style call must succeed and return a single tensor
        out = enc(x, mask, var_id=var_id, time_id=time_id)
    assert isinstance(out, torch.Tensor) and out.shape == x.shape
    print("✓ test_backward_compat_no_kwargs")


test_backward_compat_no_kwargs()

✓ test_backward_compat_no_kwargs
